In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def get_log_likelihood(scores, tokens):
    """
    Get log likelihoods for each token from the raw scores
    """
    scores = torch.stack(scores).squeeze()
    log_softmax = torch.log_softmax(scores, dim=-1)
    
    len_gen_tokens = log_softmax.shape[0]
    generated_tokens = tokens[-len_gen_tokens:]
    
    log_softmax_gen_tokens = log_softmax[range(len_gen_tokens), generated_tokens]
    return log_softmax_gen_tokens

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Prepare input
prompt = "The quick brown fox"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

# Generate tokens
outputs = model.generate(
    input_ids,
    max_new_tokens=20,
    return_dict_in_generate=True,
    output_scores=True
)

# Method 1: compute_transition_scores
transition_scores = model.compute_transition_scores(
    outputs.sequences, outputs.scores, normalize_logits=True
)

# Method 2: Manual calculation
manual_scores = []
for logits in outputs.scores:
    log_probs = torch.log_softmax(logits, dim=-1)
    manual_scores.append(log_probs)
manual_scores = torch.cat(manual_scores, dim=0)

# Method 3: get_log_likelihood function
get_log_likelihood_scores = get_log_likelihood(outputs.scores, outputs.sequences[0][input_ids.shape[1]:])

# Compare results
generated_tokens = outputs.sequences[0, input_ids.shape[1]:]
for i, (token, score1, score2, score3) in enumerate(zip(generated_tokens, 
                                                        transition_scores[0], 
                                                        manual_scores[0, generated_tokens], 
                                                        get_log_likelihood_scores)):
    print(f"Token: {tokenizer.decode(token):10s} | "
          f"Method 1: {score1:.4f} | "
          f"Method 2: {score2:.4f} | "
          f"Method 3: {score3:.4f} | "
          f"Diff (1-2): {abs(score1 - score2):.6f} | "
          f"Diff (1-3): {abs(score1 - score3):.6f}")

# Check if all methods produce approximately the same results
tolerance = 1e-5
all_close = torch.allclose(transition_scores[0], manual_scores[0, generated_tokens], atol=tolerance) and \
            torch.allclose(transition_scores[0], get_log_likelihood_scores, atol=tolerance)

print(f"\nAll methods produce the same results (within tolerance of {tolerance}): {all_close}")

/opt/conda/envs/llava/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2024-10-12 07:25:38,469] [INFO] [real_accelerator.py:161:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/opt/conda/envs/llava/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Token: es         | Method 1: -2.0427 | Method 2: -2.0428 | Method 3: -2.0428 | Diff (1-2): 0.000080 | Diff (1-3): 0.000080
Token:  are       | Method 1: -2.3210 | Method 2: -7.2213 | Method 3: -2.3212 | Diff (1-2): 4.900290 | Diff (1-3): 0.000160
Token:  a         | Method 1: -2.9873 | Method 2: -7.6411 | Method 3: -2.9874 | Diff (1-2): 4.653799 | Diff (1-3): 0.000101
Token:  great     | Method 1: -3.1264 | Method 2: -11.7374 | Method 3: -3.1265 | Diff (1-2): 8.611038 | Diff (1-3): 0.000129
Token:  way       | Method 1: -1.9558 | Method 2: -9.3969 | Method 3: -1.9558 | Diff (1-2): 7.441156 | Diff (1-3): 0.000081
Token:  to        | Method 1: -0.0827 | Method 2: -5.4036 | Method 3: -0.0829 | Diff (1-2): 5.320903 | Diff (1-3): 0.000194
Token:  get       | Method 1: -2.1680 | Method 2: -7.5871 | Method 3: -2.1682 | Diff (1-2): 5.419073 | Diff (1-3): 0.000127
Token:  a         | Method 1: -1.8458 | Method 2: -7.6411 | Method 3: -1.8458 | Diff (1-2): 5.795388 | Diff (1-3): 0.000084
Token: 

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def get_log_likelihood(scores, tokens):
    """
    Get log likelihoods for each token from the raw scores
    """
    scores = torch.stack(scores).squeeze()
    log_softmax = torch.log_softmax(scores, dim=-1)
    
    len_gen_tokens = log_softmax.shape[0]
    generated_tokens = tokens[-len_gen_tokens:]
    
    log_softmax_gen_tokens = log_softmax[range(len_gen_tokens), generated_tokens]
    return log_softmax_gen_tokens

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Prepare input
prompt = "The quick brown fox"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

# Generate tokens
outputs = model.generate(
    input_ids,
    max_new_tokens=20,
    return_dict_in_generate=True,
    output_scores=True
)

# Method 1: compute_transition_scores
transition_scores = model.compute_transition_scores(
    outputs.sequences, outputs.scores, normalize_logits=True
)

# Method 2: Manual calculation
manual_scores = []
for logits in outputs.scores:
    log_probs = torch.log_softmax(logits, dim=-1)
    manual_scores.append(log_probs)
manual_scores = torch.cat(manual_scores, dim=0)

# Method 3: get_log_likelihood function
get_log_likelihood_scores = get_log_likelihood(outputs.scores, outputs.sequences[0][input_ids.shape[1]:])

# Method 4: Log softmax and indexing
logits = torch.cat(outputs.scores, dim=0)
log_probabilities = torch.log_softmax(logits, dim=-1)
generated_tokens = outputs.sequences[0, input_ids.shape[1]:]
token_log_probs = log_probabilities[range(len(generated_tokens)), generated_tokens]

# Compare results
for i, (token, score1, score2, score3, score4) in enumerate(zip(generated_tokens, 
                                                                transition_scores[0], 
                                                                manual_scores[0, generated_tokens], 
                                                                get_log_likelihood_scores,
                                                                token_log_probs)):
    print(f"Token: {tokenizer.decode(token):10s} | "
          f"Method 1: {score1:.4f} | "
          f"Method 2: {score2:.4f} | "
          f"Method 3: {score3:.4f} | "
          f"Method 4: {score4:.4f} | "
          f"Max Diff: {max(abs(score1 - score2), abs(score1 - score3), abs(score1 - score4)):.6f}")

# Check if all methods produce approximately the same results
tolerance = 1e-5
all_close = torch.allclose(transition_scores[0], manual_scores[0, generated_tokens], atol=tolerance) and \
            torch.allclose(transition_scores[0], get_log_likelihood_scores, atol=tolerance) and \
            torch.allclose(transition_scores[0], token_log_probs, atol=tolerance)

print(f"\nAll methods produce the same results (within tolerance of {tolerance}): {all_close}")

# Calculate sequence log probability and probability for Method 4
sequence_log_prob = torch.sum(token_log_probs)
sequence_probability = torch.exp(sequence_log_prob)

print(f"\nSequence log probability: {sequence_log_prob:.4f}")
print(f"Sequence probability: {sequence_probability:.4e}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Token: es         | Method 1: -2.0427 | Method 2: -2.0428 | Method 3: -2.0428 | Method 4: -2.0428 | Max Diff: 0.000080
Token:  are       | Method 1: -2.3210 | Method 2: -7.2213 | Method 3: -2.3212 | Method 4: -2.3212 | Max Diff: 4.900290
Token:  a         | Method 1: -2.9873 | Method 2: -7.6411 | Method 3: -2.9874 | Method 4: -2.9874 | Max Diff: 4.653799
Token:  great     | Method 1: -3.1264 | Method 2: -11.7374 | Method 3: -3.1265 | Method 4: -3.1265 | Max Diff: 8.611038
Token:  way       | Method 1: -1.9558 | Method 2: -9.3969 | Method 3: -1.9558 | Method 4: -1.9558 | Max Diff: 7.441156
Token:  to        | Method 1: -0.0827 | Method 2: -5.4036 | Method 3: -0.0829 | Method 4: -0.0829 | Max Diff: 5.320903
Token:  get       | Method 1: -2.1680 | Method 2: -7.5871 | Method 3: -2.1682 | Method 4: -2.1682 | Max Diff: 5.419073
Token:  a         | Method 1: -1.8458 | Method 2: -7.6411 | Method 3: -1.8458 | Method 4: -1.8458 | Max Diff: 5.795388
Token:  little    | Method 1: -2.5784 | Method 

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import math

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Prepare input
prompt = "The quick brown fox"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

# Generate tokens
outputs = model.generate(
    input_ids,
    max_new_tokens=20,
    return_dict_in_generate=True,
    output_scores=True
)

# Method 1: compute_transition_scores
transition_scores = model.compute_transition_scores(
    outputs.sequences, outputs.scores, normalize_logits=True
)

# Method 2: Manual calculation
logits = torch.cat(outputs.scores, dim=0)
log_probs = torch.log_softmax(logits, dim=-1)

# Get generated tokens
generated_tokens = outputs.sequences[0, input_ids.shape[1]:]

# Print results
print("Token\t| Predicted Token\t| Log Probability")
print("-" * 50)
for i in range(len(generated_tokens) - 1):
    current_token = tokenizer.decode(generated_tokens[i])
    next_token = tokenizer.decode(generated_tokens[i + 1])
    log_prob = transition_scores[0, i].item()
    print(f"{current_token}\t| {next_token}\t\t| {log_prob:.4f}")

# Calculate sequence log probability
sequence_log_prob = transition_scores[0].sum().item()
sequence_prob = math.exp(sequence_log_prob)

print(f"\nSequence log probability: {sequence_log_prob:.4f}")
print(f"Sequence probability: {sequence_prob:.4e}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Token	| Predicted Token	| Log Probability
--------------------------------------------------
es	|  are		| -2.0427
 are	|  a		| -2.3210
 a	|  great		| -2.9873
 great	|  way		| -3.1264
 way	|  to		| -1.9558
 to	|  get		| -0.0827
 get	|  a		| -2.1680
 a	|  little		| -1.8458
 little	|  bit		| -2.5784
 bit	|  of		| -2.7829
 of	|  a		| -0.2421
 a	|  kick		| -2.0989
 kick	|  out		| -2.7906
 out	|  of		| -0.2548
 of	|  your		| -0.0275
 your	|  dog		| -1.4650
 dog	| .		| -3.0890
.	| 
		| -0.9451

	| 
		| -1.7745

Sequence log probability: -34.5863
Sequence probability: 9.5356e-16


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Prepare input
prompt = "The quick brown fox"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

# Generate tokens
outputs = model.generate(
    input_ids,
    max_new_tokens=5,  # Generate 5 new tokens
    return_dict_in_generate=True,
    output_scores=True  # To get the logits for each token
)

# Compute transition scores (log probabilities of the generated tokens)
transition_scores = model.compute_transition_scores(
    outputs.sequences, outputs.scores, normalize_logits=True
)

# Get the generated tokens (excluding the prompt tokens)
generated_tokens = outputs.sequences[0, input_ids.shape[1]:]

# Print the generated sequence
print(f"Generated sequence: {tokenizer.decode(generated_tokens)}")

# Display transition scores for each token
print("\nToken\t| Next Token\t| Log Probability")
print("-" * 50)

for i in range(len(generated_tokens) - 1):
    current_token = tokenizer.decode(generated_tokens[i])
    next_token = tokenizer.decode(generated_tokens[i + 1])
    log_prob = transition_scores[0, i].item()
    print(f"{current_token}\t| {next_token}\t\t| {log_prob:.4f}")

# Calculate sequence log probability
sequence_log_prob = transition_scores[0].sum().item()
sequence_prob = math.exp(sequence_log_prob)

print(f"\nTotal log probability of the generated sequence: {sequence_log_prob:.4f}")
print(f"Total probability of the generated sequence: {sequence_prob:.4e}")

/opt/conda/envs/llava/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated sequence: es are a great way

Token	| Next Token	| Log Probability
--------------------------------------------------
es	|  are		| -2.0427
 are	|  a		| -2.3210
 a	|  great		| -2.9873
 great	|  way		| -3.1264

Total log probability of the generated sequence: -12.4331
Total probability of the generated sequence: 3.9843e-06


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Prepare input
prompt = "The quick brown fox"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

# Generate tokens
outputs = model.generate(
    input_ids,
    max_new_tokens=5,  # Generate 5 new tokens
    return_dict_in_generate=True,
    output_scores=True  # To get the logits for each token
)

# Compute transition scores (log probabilities of the generated tokens)
transition_scores = model.compute_transition_scores(
    outputs.sequences, outputs.scores, normalize_logits=True
)

# Get the generated tokens (excluding the prompt tokens)
generated_tokens = outputs.sequences[0, input_ids.shape[1]:]

# Print the generated sequence
print(f"Generated sequence: {tokenizer.decode(generated_tokens)}")

# Display transition scores for each token
print("\nToken\t| Next Token\t| Log Probability")
print("-" * 50)

for i in range(len(generated_tokens) - 1):
    current_token = tokenizer.decode(generated_tokens[i])
    next_token = tokenizer.decode(generated_tokens[i + 1])
    log_prob = transition_scores[0, i].item()
    print(f"{current_token}\t| {next_token}\t\t| {log_prob:.4f}")

# Calculate sequence log probability
sequence_log_prob = transition_scores[0].sum().item()
sequence_prob = math.exp(sequence_log_prob)

print(f"\nTotal log probability of the generated sequence: {sequence_log_prob:.4f}")
print(f"Total probability of the generated sequence: {sequence_prob:.4e}")

/opt/conda/envs/llava/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated sequence: es are a great way

Token	| Next Token	| Log Probability
--------------------------------------------------
es	|  are		| -2.0427
 are	|  a		| -2.3210
 a	|  great		| -2.9873
 great	|  way		| -3.1264

Total log probability of the generated sequence: -12.4331
Total probability of the generated sequence: 3.9843e-06
